In [ ]:
# 数据获取与示例构造
# 尝试从 logs/uncertainty_dump.pt 读取真实运行日志；若不存在则生成演示数据

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path

plt.rcParams['figure.figsize'] = (6, 4)

LOG_PATH = Path('logs/uncertainty_dump.pt')


def load_or_mock(path: Path):
    if path.exists():
        payload = torch.load(path, map_location='cpu')
        print(f"Loaded log from {path}")
        return payload
    print(f"{path} not found, generating mock data …")
    num_steps = 6
    sigma_history = []
    traces = []
    for i in range(num_steps):
        base = torch.diag(torch.tensor([0.02, 0.02, 0.08]) ** 2)
        decay = torch.exp(-0.4 * i)
        Sigma = base * decay + 1e-5 * torch.rand(3, 3)
        Sigma = 0.5 * (Sigma + Sigma.t())
        sigma_history.append(Sigma)
        traces.append(torch.trace(Sigma).item())
    var_maps = torch.rand(num_steps, 16, 16, 4) * torch.linspace(1.0, 0.2, num_steps)[:, None, None, None]
    sqrt_info = torch.rand(num_steps, 512, 4)
    ate = np.linspace(0.08, 0.02, num_steps) + 0.005 * np.random.randn(num_steps)
    point_before = torch.randn(512, 3)
    point_after = point_before + 0.05 * torch.randn(512, 3)
    return {
        'sigma_history': torch.stack(sigma_history),
        'var_maps': var_maps,
        'sqrt_info': sqrt_info,
        'ate': torch.tensor(ate),
        'trace_sigma': torch.tensor(traces),
        'point_before': point_before,
        'point_after': point_after,
    }

payload = load_or_mock(LOG_PATH)



In [ ]:
# 1. 协方差椭球 (前三次融合)
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

sigmas = payload['sigma_history']
levels = min(3, sigmas.shape[0])
colors = ['#ff7f0e', '#1f77b4', '#2ca02c']

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

for i in range(levels):
    S = sigmas[i].numpy()
    vals, vecs = np.linalg.eigh(S)
    radii = np.sqrt(vals)
    u = np.linspace(0.0, 2 * np.pi, 20)
    v = np.linspace(0.0, np.pi, 20)
    x = radii[0] * np.outer(np.cos(u), np.sin(v))
    y = radii[1] * np.outer(np.sin(u), np.sin(v))
    z = radii[2] * np.outer(np.ones_like(u), np.cos(v))
    xyz = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    xyz = xyz @ vecs.T
    ax.plot_trisurf(
        xyz[:, 0], xyz[:, 1], xyz[:, 2],
        color=colors[i % len(colors)], alpha=0.25, linewidth=0.2
    )

ax.set_title('Covariance ellipsoids (iterations 0-2)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.show()



In [ ]:
# 2. 残差方差热图 (射线/距离)
var_maps = payload['var_maps']  # shape: steps x H x W x 4
frame_id = 0
ray_var = var_maps[frame_id, :, :, :3].mean(-1)
dist_var = var_maps[frame_id, :, :, 3]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(ray_var.numpy(), cmap='viridis')
axes[0].set_title('Ray variance (avg of 3 dims)')
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(dist_var.numpy(), cmap='plasma')
axes[1].set_title('Distance variance')
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.show()



In [ ]:
# 3. 权重分布 (sqrt_info)
sqrt_info = payload['sqrt_info']  # steps x N x dim
step_a, step_b = 0, min(3, sqrt_info.shape[0]-1)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(sqrt_info[step_a].flatten().numpy(), bins=40, alpha=0.6, label=f'step {step_a}')
ax.hist(sqrt_info[step_b].flatten().numpy(), bins=40, alpha=0.6, label=f'step {step_b}')
ax.set_title('Distribution of sqrt_info weights')
ax.set_xlabel('sqrt_info'); ax.set_ylabel('count'); ax.legend()
plt.show()



In [ ]:
# 4. 协方差特征值随迭代变化
sigmas = payload['sigma_history']
eigs = []
for S in sigmas:
    eigs.append(np.linalg.eigvalsh(S.numpy()))
eigs = np.stack(eigs)
fig, ax = plt.subplots(figsize=(6, 4))
for i in range(3):
    ax.plot(eigs[:, i], marker='o', label=f'eig{i}')
ax.set_yscale('log')
ax.set_xlabel('iteration'); ax.set_ylabel('eigenvalue (log)')
ax.set_title('Covariance spectrum convergence')
ax.legend()
plt.show()



In [ ]:
# 5. ATE / trace(Σ) 关系
trace_sigma = payload['trace_sigma'].numpy()
ate = payload['ate'].numpy()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(trace_sigma, label='trace(Sigma)', marker='o')
ax.set_ylabel('trace(Sigma)'); ax.set_xlabel('iteration')
ax2 = ax.twinx()
ax2.plot(ate, color='tab:red', label='ATE', marker='s')
ax2.set_ylabel('ATE [m]')
ax.set_title('ATE vs average covariance trace')
fig.legend(loc='upper right')
plt.show()



In [ ]:
# 6. 融合前后点云差分
pts_before = payload['point_before'].numpy()
pts_after = payload['point_after'].numpy()
delta = pts_after - pts_before

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts_before[:200, 0], pts_before[:200, 1], pts_before[:200, 2],
           s=6, alpha=0.4, label='before')
ax.scatter(pts_after[:200, 0], pts_after[:200, 1], pts_after[:200, 2],
           s=6, alpha=0.4, label='after')
ax.set_title('Point cloud before/after Bayesian fusion (subset)')
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(np.linalg.norm(delta, axis=1), bins=40)
ax.set_title('Per-point displacement magnitude')
ax.set_xlabel('||ΔX||'); ax.set_ylabel('count')
plt.show()

